# 1단계 · 밑바닥부터 만드는 초소형 GPT

이 노트북에서는 라이브러리에 의존하지 않고 **트랜스포머 언어모델을 직접 구현**합니다.
목표는 성능이 아니라 **원리 이해**입니다. 다음을 직접 만듭니다:

1. 글자 단위 토크나이저 (텍스트 ↔ 숫자)
2. 셀프 어텐션 / 트랜스포머 블록
3. 학습 루프
4. 텍스트 생성

> 실행 전: **런타임 → 런타임 유형 변경 → T4 GPU** 선택 (CPU로도 돌아가지만 느립니다)


## 0. 준비 — PyTorch 임포트
Colab에는 PyTorch가 기본 설치되어 있습니다.


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(1337)
print('device:', device)

## 1. 학습 데이터 준비

언어모델은 '다음 글자 맞히기'로 언어를 배웁니다. 작은 한국어 말뭉치를 만들겠습니다.
(데이터가 작아 모델이 금방 외워버리는데, 이는 '학습이 되고 있다'는 걸 눈으로 확인하기 좋습니다.)

원하면 아래 `text` 변수에 여러분의 텍스트(블로그 글, 일기 등)를 넣어 바꿔도 됩니다.


In [ ]:
text = (
'''인공지능은 데이터를 학습해 패턴을 찾아내는 기술이다.
언어모델은 다음에 올 글자를 예측하면서 언어를 배운다.
벡터 데이터베이스는 의미가 비슷한 문장을 빠르게 찾아준다.
임베딩은 문장을 숫자 벡터로 바꾼 표현이다.
검색 증강 생성은 관련 문서를 먼저 찾아 답변에 활용한다.
셀프 어텐션은 문장 속 단어들이 서로를 참고하게 만든다.
트랜스포머는 현대 언어모델의 핵심 구조다.
파인튜닝은 큰 모델을 내 데이터로 미세조정하는 일이다.
코사인 유사도는 두 벡터가 이루는 각도로 유사성을 잰다.
토큰은 모델이 읽는 가장 작은 단위다.
'''
) * 50  # 작은 말뭉치를 반복해 학습용으로 키운다

chars = sorted(list(set(text)))
vocab_size = len(chars)
print('전체 글자 수:', len(text))
print('고유 글자(어휘) 수:', vocab_size)
print('어휘:', ''.join(chars))

## 2. 토크나이저 — 글자 ↔ 숫자

모델은 숫자만 다룹니다. 각 글자에 정수 번호를 매깁니다(글자 단위 토크나이저).
실제 LLM은 '단어 조각' 단위(BPE)를 쓰지만 원리는 같습니다.


In [ ]:
stoi = {ch: i for i, ch in enumerate(chars)}  # 글자 -> 정수
itos = {i: ch for i, ch in enumerate(chars)}  # 정수 -> 글자
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)

print(encode('인공지능'))
print(decode(encode('인공지능')))

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print('학습:', len(train_data), '검증:', len(val_data))

## 3. 하이퍼파라미터

모델 크기와 학습 설정입니다. 숫자를 키우면 똑똑해지지만 느려집니다. 먼저 이대로 돌려보세요.


In [ ]:
block_size = 64    # 한 번에 보는 글자 수(문맥 길이)
batch_size = 32    # 한 번에 학습하는 묶음 수
n_embd     = 128   # 임베딩 차원
n_head     = 4     # 어텐션 헤드 수
n_layer    = 4     # 트랜스포머 블록 수
dropout    = 0.1
learning_rate = 3e-4
max_iters  = 3000
eval_interval = 300

## 4. 배치 만들기

무작위 위치에서 `block_size`만큼 잘라, 입력 x와 '한 칸 뒤로 민' 정답 y를 만듭니다.
모델은 x를 보고 y(=다음 글자)를 맞히도록 배웁니다.


In [ ]:
def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch('train')
print('입력 x:', xb.shape, '| 정답 y:', yb.shape)

## 5. 셀프 어텐션 — 핵심 부품

각 글자를 **Query(질문) / Key(열쇠) / Value(값)** 벡터로 바꾼 뒤,
Query·Key 유사도로 '누구를 얼마나 참고할지' 가중치를 만들고 Value를 가중합합니다.
`tril` 마스크로 **미래 글자는 못 보게** 막습니다(다음 글자를 미리 보면 안 되니까요).


In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)    # (B,T,hs)
        q = self.query(x)  # (B,T,hs)
        # 유사도(어텐션 점수) 계산 후 스케일링
        w = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5  # (B,T,T)
        w = w.masked_fill(self.tril[:T, :T] == 0, float('-inf'))  # 미래 차단
        w = F.softmax(w, dim=-1)
        w = self.dropout(w)
        v = self.value(x)
        return w @ v  # (B,T,hs)

class MultiHead(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

## 6. 트랜스포머 블록 & 전체 모델

블록 = (멀티헤드 어텐션) + (작은 신경망 FFN), 각각 잔차연결과 LayerNorm으로 감쌉니다.
이 블록을 `n_layer`개 쌓고, 마지막에 '다음 글자 확률'을 내는 선형층을 둡니다.


In [ ]:
class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHead(n_head, head_size)
        self.ff = FeedForward()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))   # 잔차연결
        x = x + self.ff(self.ln2(x))
        return x

class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, n_embd)   # 글자 임베딩
        self.pos_emb = nn.Embedding(block_size, n_embd)   # 위치 임베딩
        self.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.tok_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=device))
        x = tok + pos
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)  # (B,T,vocab)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_id), dim=1)
        return idx

model = TinyGPT().to(device)
print('파라미터 수:', sum(p.numel() for p in model.parameters())/1e6, 'M')

## 7. 학습 — 다음 글자 맞히기

loss(손실)가 줄어들면 모델이 한국어 패턴을 배우고 있다는 뜻입니다.
T4 GPU에서 몇 분이면 끝납니다.


In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(50)
        for k in range(50):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for it in range(max_iters + 1):
    if it % eval_interval == 0:
        l = estimate_loss()
        print(f"step {it:4d} | train loss {l['train']:.3f} | val loss {l['val']:.3f}")
    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print('학습 완료!')

## 8. 텍스트 생성 — 직접 만든 모델로!

빈 문맥에서 시작해 모델이 한 글자씩 이어 씁니다. 데이터가 작아 학습 문장과 비슷하게 나올 거예요.
이게 바로 '내가 만든 가장 작은 LLM'입니다. 🎉


In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated = model.generate(context, max_new_tokens=300, temperature=0.8)
print(decode(generated[0].tolist()))

In [ ]:
# 특정 문장으로 시작해보기 (프롬프트)
prompt = '인공지능은'
ctx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
out = model.generate(ctx, max_new_tokens=200, temperature=0.5)
print(decode(out[0].tolist()))

## 정리 & 다음 단계

방금 토크나이저 · 셀프어텐션 · 트랜스포머 블록 · 학습루프 · 생성을 **직접** 만들었습니다.
진짜 LLM도 구조는 동일하고, 차이는 **데이터 양 · 모델 크기 · 토크나이저(BPE)** 뿐입니다.

개인이 from-scratch로 쓸만한 한국어 챗봇을 만드는 건 비현실적이므로,
다음 노트북 **`02_finetune_lora.ipynb`** 에서 진짜 오픈모델(Qwen)을 가져와
내 한국어 Q&A 데이터로 **파인튜닝**해 실제로 쓸 수 있는 챗봇을 만듭니다.

### 실험해보기
- `max_iters`, `n_layer`, `n_embd`를 키우면 어떻게 달라질까?
- `temperature`를 0.2 / 1.2로 바꾸면 생성 결과가 어떻게 변할까?
- `text`에 내 글을 넣으면 내 말투를 흉내 낼까?
